In [ ]:
%pylab inline
import eucare as ec

In [ ]:
from rdp import rdp

def profile_function(x):
    # should be 0 at 0 and defined on [0, 1]
#     a = 0.6
#     return np.where(x > a,
#                     np.sqrt(np.clip((1-a)**2-(1-x)**2, a_min=0, a_max=1)),
#                     0)
    return 1.3*np.sqrt(1-(1-x)**2)
#     return (1-(1-x)**2)

t = np.linspace(0, 1, 1000)

# height along the crossection
y = profile_function(t)

plt.plot(t, y)
plt.gca().set_aspect(1)
plt.show();

dy = y[1:] - y[:-1]
dt = t[1:] - t[:-1]
l = np.concatenate([[0], np.cumsum(np.sqrt(dy**2 + dt**2))])

tl = np.stack([t, l], axis=-1)
tl = rdp(tl, 0.0001)
print(tl.shape)
t, l = tl.T

shrink_factor = 1/l[-1]
print(f'shrink factor: {shrink_factor}')
l *= shrink_factor
t *= shrink_factor


# plt.plot(t, l)
plt.plot(l, t)
plt.scatter(l, t, marker='.')
plt.gca().set_aspect(1)

In [ ]:
# from scipy.interpolate import CubicSpline, BSpline
# x = np.arange(10)
# y = np.sin(x)
# cs = CubicSpline(x, y)
# xs = np.arange(-0.5, 9.6, 0.1)
# fig, ax = plt.subplots()
# ax.plot(x, y, 'o', label='data')
# cs.antiderivative()(xs)

In [ ]:
def reverse_mapping(d):
    return {value: key for key, value in d.items()}

def from_pre(obj):
    return 'pre_conway' in obj.attributes

def line_intersection(line1, line2):
    diff = np.stack([l[0] - l[1] for l in (line1, line2)])
    
    div = np.linalg.det(diff)
    if div == 0:
        raise Exception('lines do not intersect')

    d = np.array([np.linalg.det(l) for l in (line1, line2)])
    return np.array([np.linalg.det(np.stack([d, dif])) for dif in diff.T]) / div

def pseudo_incenter(f):
    """
    If f has an incenter, this is it.
    Otherwise, this is the mean of all the incenters of triangles with corners being a set of adjacent corners of f. 
    """
    ps = np.array([v['pos'] for v in f.vertex_iter()])
    lengths = np.linalg.norm(np.roll(ps, 1, axis=0) - np.roll(ps, 2, axis=0), axis=-1)
    incenter =  np.sum(lengths[:, None] * ps, axis=0) / np.sum(lengths)
    return incenter

def make_intersecting_cylinders(G, r=1):
    for v in G.vertices:
        if 'pre_conway' in v:
            del v['pre_conway']

    for f in list(G.faces):
        f['midpoint'] = pseudo_incenter(f)

    if r != 1:
        G = ec.conway.expand_graph((1-r) * shrink_factor / (1 - (1-r) * (1-shrink_factor)))(G, delete_on_border=False)
        fs = [f for f in G.faces if 'pre_conway' in f]
        for f in fs:
            f['midpoint'] = pseudo_incenter(f)
    else:
        fs = G.faces

    # for f in fs:
    #     f['color_key'] = (1, 0, 0)
    # G.show()

    G, (v_map, h_map, f_map_f) = G.copy(return_mappings=True)

    v_map, h_map, f_map = [reverse_mapping(m) for m in (v_map, h_map, f_map_f)]
    vertex_pairs_to_halfedges = {(h.orig, h.dest): h_map[h] for h in G.halfedges}

    G = ec.conway.lace_graph(0.5, join=True)(G, faces=[f_map_f[f] for f in fs], 
                                             delete_inner_border=True,
                                             delete_on_border=False,
                                            copy_graph=False)
    G.delete_subset([f for f in G.faces if f.any_side not in G.halfedges])
    G.check_consistency()
    # construct map from pairs of new vertices to original halfedges

    # construct map from curved halfedges to original halfedges

    vs = [v for v in G.vertices if from_pre(v) and isinstance(v['pre_conway'], ec.half.Vertex)]
    print(len(vs), len([v for v in G.halfedges if from_pre(v)]), len([v for v in G.faces if from_pre(v)]))
    curved_halfedges = []
    for v in vs:
        for h in v.outgoing_iter():
            if h.dest in vs:
                h['color_key'] = h.rev['color_key'] = (0, 0, 0)
                continue
            elif h.nex.dest in vs:
                v2 = h.nex.dest
            else:
                h['color_key'] = h.rev['color_key'] = (0, 0, 1)
                continue
            try:
                h_orig = vertex_pairs_to_halfedges[(v2['pre_conway'], v['pre_conway'])]
            except KeyError:
                continue
            # postions of vertices
            p = v['pos']
            p2 = v2['pos']

            # center of original facef
            c = h_orig.face.midpoint()
            hc = ec.base.project_to_line(np.stack([p, p2]), c)

            # altitude from hc to c and segment of side from p to hc define coordinate system
            alt = c - hc
            side = hc - p

            curve = (side[:, None] * l + alt[:, None] * t).T + p
            extra_point = line_intersection(
                np.stack([p2, c]),
                np.stack([curve[-1], curve[-1] + c - p])
            )
            curve = np.concatenate([curve, extra_point[None]])
            
            h.dest['pos'] = (h.dest['pos'] + c) / 2
            h['curve_pos'] = curve
            h.rev['curve_pos'] = curve[::-1]
            h.dest['pos'] = curve[-1]
            h['color_key'] = h.rev['color_key'] = (1, 0, 0)

    for h in G.halfedges:
        if 'color_key' not in h:
            h['color_key'] = (0, 0, 1)
    
#     G.show(line_width=0.03, render_faces=False, render_vertices=False, height=1000)

    return G


def convert_to_triangle_twist(G, v):
    # delete curved creases
    G.delete_subset([h for h in v.outgoing_iter() if h['color_key'] == (1, 0, 0)])
    # set_color((0, 1, 0), h0)
    tasks = []
    for h in list(v.outgoing_iter()):
        v2 = h.rev.nex.nex.dest
        set_color((0, 1, 0), v2)
        direction = v['pos'] - h.rev.nex.dest['pos'] # pointing to v
        f = h.rev.face
        pos = ec.base.line_intersection(
            [v2['pos'], v2['pos'] + direction],
            [h.orig['pos'], h.dest['pos']]
        )
        tasks.append((h, f, v2, pos))

    # add first set of creases
    for h, f, v2, pos in tasks:
        _, v3 = G.subdivide_edge(h, pos=pos, color_key=(1, 1, 0))
        h2, _ = G.subdivide_face(f, v2, v3)
        set_color((1, 0, 0), h2)

    # add twist face 
    for h in v.outgoing_iter():
        h2, _ = G.subdivide_face(h.face, h.dest, h.pre.orig)
        set_color((0, 0, 1), h2)
    # finally, delete central vertex
    G.delete_subset(v)
    
def convert_all_to_triangle_twists(G):
    vs = [v for v in G.vertices if v.order() == 6 and not v.on_border()]
    for v in vs:
        convert_to_triangle_twist(G, v)


In [ ]:
#TODO: render backlit model

In [ ]:
import os

r = 1

path = 'graphs/irregular2.heg' #'graphs/circlepack/egg_b.heg'
kis = True

G = ec.io.load_graph(path)
for f in G.faces:
    f['midpoint'] = f.pseudo_incenter()
    
if kis:
    G = ec.conway.kis_graph()(G, delete_on_border=True)
# G.show()
# G = ec.io.load_graph('graphs/irregular2.heg')
# G = ec.example_graphs.from_tiles(ec.example_tilesets.curved_platonic(3, 6), rings=0)
# G = ec.conway.kis_graph()(G, delete_on_border=True)
# G = ec.example_graphs.from_tiles(ec.example_tilesets.u2_4_6_12__3_4_6_4(), rings=2)

# ps, vs = G.get_position_view()
# k = ps.copy()
# k = np.array([complex(*ki) for ki in k]) * 0.1j
# ps[:] = np.stack([k.real, k.imag], axis=-1)

# # cut along positive real axis
# es = list(G.halfedges)
# eps = 1e-6
# es = [e for e in es if e.orig['pos'][1] > 2*eps and e.dest['pos'][1] < eps and e.orig['pos'][0] < 0]
# vs = [v for v in G.vertices if v['pos'][0] < -eps and np.abs(v['pos'][1]) < eps]
# G.delete_subset(vs, es)

# ps, vs = G.get_position_view()

# k = ps.copy()
# k = np.array([complex(*ki) for ki in k])
# # k -= np.mean(k)

# mask = np.abs(k) < eps
# print(np.sum(mask))
# k = k**(6/5)
# k[mask] = 0

# k = np.stack([k.real, k.imag], axis=-1)
# ps[:] = k
# ps[:] *= np.std(np.linalg.norm(ps, axis=-1)) * 100
# G = ec.overlap.remove_duplicates(G)

# G = ec.conway.dual_graph()(G)
ec.overlap.optimize_rotation(G)
CP = make_intersecting_cylinders(G, r=r)

convert_all_to_triangle_twists(CP)

for f in G.faces:
    f['midpoint'] = f.pseudo_incenter()
folded = ec.conway.join_graph()(G.copy(), delete_on_border=False)
folded.show()
ps, _ = folded.get_position_view()
# ps[:] *= shrink_factor

result = {
    'CP': CP,
    'folded_state': folded,
    'folded_view_top': folded,
}
print('saving result')
# bbox = (25, 20)
bbox = (65, 48)
ec.overlap.save_results(result, f"nice_images/intersecting_cylinders/{os.path.basename(path).split('.')[0]+('' if not kis else '_kis')}", bbox=bbox, 
                        render_settings=dict(line_width=0.0005, render_vertices=False, height=2000))

In [ ]:
def set_color(color, *objs):
    for obj in objs:
        if isinstance(obj, (ec.half.Face, ec.half.HalfEdge, ec.half.Vertex)):
            obj['color_key'] = color
            if isinstance(obj, (ec.half.HalfEdge)):
                obj.rev['color_key'] = color
        else:
            set_color(color, *obj)

In [ ]:
from glob import glob
list(map(print, glob('./*.heg')))
G = ec.io.load_graph('graphs/circlepack/octagon2sym.heg')
G.show()